In [ ]:
import sys
sys.path.extend(["../../../"])
import numpy as np 
import matplotlib.pyplot as plt

import os
import imageio.v2 as imageio
from IPython.display import Markdown, display, Video
from io import BytesIO

from utils import squared_loss_function
from plot_utils import Arrow3D

# Gradient Descent with Momentum

In [ ]:
np.random.seed(1)
N_data = 3
x_data = np.random.randn(N_data,2)* np.array([0.1,0.1]) + np.array([15,4]) 
t_data = np.array([[0,0],[0,0],[0,0]])

In [ ]:
x_data = np.stack([
    np.column_stack((x_data[:,0] ,np.zeros(N_data))),
    np.column_stack((np.zeros(N_data),x_data[:,1]))
], axis=1 )
t_data = np.reshape(t_data,(N_data,-1,1))

In [ ]:
print(x_data)

In [ ]:
## To do so we need a mesh
N_points_domain = 100
w1, w2 = np.meshgrid(np.linspace(-1,1,N_points_domain),np.linspace(-1,5,N_points_domain))

# reshape for neural network
w_range = np.hstack((np.reshape(w1, (N_points_domain**2,1)),np.reshape(w2, (N_points_domain**2,1))))

w_range = w_range[:,np.newaxis,:,np.newaxis]

In [ ]:
# compute linear projection at all pairs of points
#y_pred = np.matmul(x_data,w_range)
y_pred = x_data @ w_range

# compute loss
expected_squared_loss_mesh = np.sum(squared_loss_function(t_data, y_pred), axis = (1,2,3))
expected_squared_loss_mesh = np.reshape(expected_squared_loss_mesh, (N_points_domain,N_points_domain))

In [ ]:
class SGD:
    def __init__(self,init_lr):
        self.lr = init_lr
    def step(self,w, grad):
        return w -self.lr*grad

class SGDMomentum:
    def __init__(self,init_lr, mmu):
        self.lr = init_lr
        self.mmu = mmu
        self.grad_mmu = np.array([[0],[0]])
    
    def step(self, w, grad):
        self.grad_mmu = self.mmu * self.grad_mmu - self.lr * grad
        return w + self.grad_mmu

def grad(X,T,W):
    return np.sum( X @ (-2 * (T-X@W)),axis = 0)
    

In [ ]:
plt.close("all")


epochs = 40

# Parametros para modificacion de la media  np.array([15,4])

lr = 0.0001 # lr 0.0015 oscillates vainila
             # lr 0.00145 tarda en converger
             # lr 0.0012 va similar pero tarda un poco más
             #  0.0001 con mmu 0.5 queda guapisimo
           
mmu = 0.9 # mmu 0.1 va bien
          # mmu 0.9 oscila y tarda 
          # mmu 0.5 va perfecto.
          # mmu 0.2 con lr 0.00145 tipico dibujo de momentum, con lr 0.00148 vanilia oscila
    
'''
# Parametros para modificacion de la media  np.array([15,4])
lr = 0.00145  # lr = 0.00145 oscila bastante SGD combinado con mmu 0.9 se aprecia buen efecto
             # lr = 0.0016 el vainila oscila y el momentum con 0.9 y 0.5 va, oscilando más 0.9
             # lr = 0.0010 y mmu = 0.9 se ve el underdamping
             # lr = 0.0010 y mmu = 0.5 se reduce el underdamping
             # lr = 0.0010 y mmu = 0.2 se reduce el underdamping
mmu = 0.35
'''

video_filename = "/tmp/aux4.mp4"
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

lrs = [0.00145]
mmus = [0.9]

fig = plt.figure(figsize=(10, 10))

# Dictionary to store variables for each combination of lr and mmu
c = 1
optimizers_data = {}
for lr in lrs:
    for mmu in mmus:
        optimizers_data[(lr, mmu)] = {
            "w": np.array([[-0.75], [4]]),
            "w_mmu": np.array([[-0.75], [4]]),
            "trajectory_sgd": [],
            "trajectory_sgdmmu": [],
            "opt_mmu": SGDMomentum(lr, mmu),
            "opt_sgd": SGD(lr),
            "ax" : fig.add_subplot(len(lrs), len(mmus), c, projection="3d")
        }
        c += 1

for e in range(epochs):
    for lr in lrs:
        for mmu in mmus:
            data = optimizers_data[(lr, mmu)]
            w, w_mmu = data["w"], data["w_mmu"]
            trajectory_sgd, trajectory_sgdmmu = data["trajectory_sgd"], data["trajectory_sgdmmu"]
            opt_sgd, opt_mmu = data["opt_sgd"], data["opt_mmu"]

            # Compute loss
            loss = np.sum(squared_loss_function(t_data, x_data @ w))
            loss_mmu = np.sum(squared_loss_function(t_data, x_data @ w_mmu))

            # Compute gradient and update parameters
            grad_w = grad(x_data, t_data, w)
            grad_w_mmu = grad(x_data, t_data, w_mmu)

            w_n = opt_sgd.step(w, grad_w)
            w_mmu_n = opt_mmu.step(w_mmu, grad_w_mmu)

            loss_n = np.sum(squared_loss_function(t_data, x_data @ w_n))
            loss_mmu_n = np.sum(squared_loss_function(t_data, x_data @ w_mmu_n))

            # Clear the plot and redraw
            ax = data["ax"]
            ax.cla()
            ax.plot_surface(w1, w2, expected_squared_loss_mesh, cmap = "gray")
            ax.view_init(elev=20, azim=-90)

            # Plot previous trajectories
            for coordinates in trajectory_sgd:
                old_w, old_w_n, old_loss, old_loss_n = coordinates
                a = Arrow3D([old_w[0, 0], old_w_n[0, 0]], [old_w[1, 0], old_w_n[1, 0]], [old_loss, old_loss_n],
                            mutation_scale=20, lw=2.0, arrowstyle="-", color="C1")
                ax.add_artist(a)

            for coordinates in trajectory_sgdmmu:
                old_w, old_w_n, old_loss, old_loss_n = coordinates
                a = Arrow3D([old_w[0, 0], old_w_n[0, 0]], [old_w[1, 0], old_w_n[1, 0]], [old_loss, old_loss_n],
                            mutation_scale=20, lw=2.0, arrowstyle="-", color="C2")
                ax.add_artist(a)

            # Plot gradient direction
            a = Arrow3D([w[0, 0], w_n[0, 0]], [w[1, 0], w_n[1, 0]], [loss, loss_n],
                        mutation_scale=20, lw=2.0, arrowstyle="-|>", color="C1", label='SGD Vanilla')
            ax.add_artist(a)

            a = Arrow3D([w_mmu[0, 0], w_mmu_n[0, 0]], [w_mmu[1, 0], w_mmu_n[1, 0]], [loss_mmu, loss_mmu_n],
                        mutation_scale=20, lw=2.0, arrowstyle="-|>", color="C2", label='SGD + Momentum')
            ax.add_artist(a)

            # Save trajectories
            trajectory_sgd.append((w, w_n, loss, loss_n))
            trajectory_sgdmmu.append((w_mmu, w_mmu_n, loss_mmu, loss_mmu_n))

            # Update dictionary
            data["w"], data["w_mmu"] = w_n, w_mmu_n

            ax.legend()
            ax.set_title(f'lr {lr} mmu: {mmu}')

    # Save frame for video
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    buf.seek(0)
    frame = imageio.imread(buf)
    writer.append_data(frame)

writer.close()
plt.close("all")

In [ ]:
display(Video(data=video_filename, embed=True))
os.remove(video_filename)